# 07 - Multi-Seed Final Evaluation

Aggregate the completed LightGCN, EmerG, DGD, and selected DGD-ablation result
bundles across seeds. This notebook does not retrain models; notebooks 03-06
own training and evaluation. Notebook 07 verifies those bundles, computes
seed-level summaries, flags missing seed coverage, and publishes the final
MovieLens-1M evaluation bundle for tables and figures.

## Notebook Linkage and Work Plan

**Inputs:** notebook-02 protocol identity plus result manifests from notebooks
03-06. Each model bundle already contains validation-selected thresholds and
held-out evaluation metrics. The DGD-ablation bundle also records which variant
was selected using validation only.

**What this notebook does:**
1. Discover local/Kaggle result manifests and verify required artifact hashes.
2. Build a run registry keyed by model family, model label, seed, and bundle ID.
3. Load phase-level evaluation metrics and validation thresholds from every run.
4. Aggregate mean, standard deviation, and standard error by model and phase.
5. Export run registry, per-run metrics, aggregate metrics, prediction-artifact
   registry, reproducibility audit, and a manifest for notebook 08.

In [1]:
from __future__ import annotations

import hashlib
import importlib.util
import json
import os
import platform
import shutil
import uuid
from datetime import datetime, timezone
from pathlib import Path
from typing import Any, Iterable

REQUIRED_PACKAGES = ["numpy", "pandas", "IPython"]
MISSING_PACKAGES = [
    package for package in REQUIRED_PACKAGES if importlib.util.find_spec(package) is None
]
if MISSING_PACKAGES:
    raise RuntimeError(
        "Notebook 07 requires these packages in the active kernel: "
        + ", ".join(MISSING_PACKAGES)
        + ". Run it in the project ML/Kaggle environment used for evaluation notebooks."
    )

import numpy as np
import pandas as pd
from IPython.display import Markdown, display


def show_records(records: Iterable[dict[str, Any]]) -> None:
    display(pd.DataFrame(list(records)))


def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def reject_json_constant(value: str) -> None:
    raise ValueError(f"Non-finite JSON constant is not allowed: {value}")


def strict_json_loads(payload: str | bytes) -> Any:
    return json.loads(payload, parse_constant=reject_json_constant)


def strict_json_dumps(value: Any, **kwargs: Any) -> str:
    return json.dumps(value, allow_nan=False, **kwargs)


def project_root(start: Path) -> Path:
    override = os.environ.get("COLDSTART_PROJECT_ROOT")
    if override:
        return Path(override).expanduser().resolve()
    for candidate in (start, *start.parents):
        if (candidate / ".git").exists():
            return candidate.resolve()
    return start.resolve()


def resolve_inside(root: Path, relative_path: str) -> Path:
    resolved_root = root.resolve()
    resolved = (resolved_root / relative_path).resolve()
    resolved.relative_to(resolved_root)
    return resolved


EXECUTION_CONTEXT = "kaggle" if Path("/kaggle/input").exists() else "local"
PROJECT_ROOT = project_root(Path.cwd())
WORKSPACE_ROOT = Path(
    os.environ.get(
        "COLDSTART_WORKSPACE_ROOT",
        "/kaggle/working" if EXECUTION_CONTEXT == "kaggle" else PROJECT_ROOT / ".notebook",
    )
).expanduser().resolve()
INPUT_ROOT = Path(
    os.environ.get(
        "COLDSTART_INPUT_ROOT",
        "/kaggle/input" if EXECUTION_CONTEXT == "kaggle" else PROJECT_ROOT / "data",
    )
).expanduser().resolve()
ARTIFACT_ROOT = Path(
    os.environ.get("COLDSTART_ARTIFACT_ROOT", WORKSPACE_ROOT / "artifacts")
).expanduser().resolve()
OUTPUT_ROOT = ARTIFACT_ROOT / "evaluations" / "ml-1m" / "multiseed-v1"
PROTOCOL_RELATIVE_MANIFEST = Path("protocols/ml-1m/coldstart-v1/manifest.json")
EXPECTED_PROTOCOL_SCHEMA = "ml1m-coldstart-v1"
TARGET_SEEDS = [
    int(seed)
    for seed in os.environ.get("COLDSTART_MULTI_SEEDS", "2025,7788,9999,3407,4517").split(",")
    if seed.strip()
]
TARGET_SEED_SET = set(TARGET_SEEDS)
PHASE_ORDER = ["Cold", "Warm A", "Warm B", "Warm C"]
EXPECTED_FAMILIES = {
    "lightgcn": {"schema": "lightgcn-baseline-v1", "label": "LightGCN"},
    "emerg": {"schema": "emerg-baseline-v1", "label": "EmerG"},
    "dgd": {"schema": "dgd-model-v1", "label": "DGD"},
    "dgd_ablation": {"schema": "dgd-ablation-v1", "label": "DGD-Ablation"},
}
SCHEMA_TO_FAMILY = {value["schema"]: key for key, value in EXPECTED_FAMILIES.items()}
RUN_CONFIG = {
    "schema_version": "multiseed-evaluation-v1",
    "target_seeds": TARGET_SEEDS,
    "phase_order": PHASE_ORDER,
    "expected_families": EXPECTED_FAMILIES,
    "selection_policy": "aggregate verified result bundles; do not tune on evaluation metrics",
}
RUN_CONFIG_SHA256 = hashlib.sha256(
    strict_json_dumps(RUN_CONFIG, sort_keys=True, separators=(",", ":")).encode()
).hexdigest()

show_records(
    [
        {
            "execution_context": EXECUTION_CONTEXT,
            "python": platform.python_version(),
            "artifact_root": str(ARTIFACT_ROOT),
            "output_root": str(OUTPUT_ROOT),
            "target_seeds": TARGET_SEEDS,
            "run_config_sha256": RUN_CONFIG_SHA256,
        }
    ]
)

,execution_context,python,artifact_root,output_root,target_seeds,run_config_sha256
0,kaggle,3.12.13,/kaggle/working/artifacts,/kaggle/working/artifacts/evaluations/ml-1m/mu...,"[2025, 7788, 9999, 3407, 4517]",34dd820eba950c6d1a503a15e02e49436d4f75d86ee760...


In [2]:
METRIC_COLUMNS = ["accuracy", "precision", "recall", "f1", "roc_auc", "predicted_positive_rate"]
BASE_REQUIRED_SOURCE_ARTIFACTS = (
    "evaluation_metrics",
    "validation_thresholds",
    "evaluation_predictions",
)


def manifest_schema(manifest: dict[str, Any]) -> str | None:
    return manifest.get("model_schema_version") or manifest.get("ablation_schema_version")


def manifest_status(manifest: dict[str, Any]) -> str | None:
    return manifest.get("model_status") or manifest.get("ablation_status")


def candidate_manifest_paths() -> list[Path]:
    candidates: list[Path] = []
    roots = [PROJECT_ROOT / ".notebook" / "artifacts", ARTIFACT_ROOT]
    if INPUT_ROOT.is_dir():
        roots.append(INPUT_ROOT)
    for root in roots:
        if root.is_dir():
            candidates.extend(root.rglob("manifest.json"))
    unique: list[Path] = []
    seen: set[str] = set()
    for path in sorted(candidates):
        key = str(path.resolve())
        if key not in seen:
            seen.add(key)
            unique.append(path.resolve())
    return unique


def artifact_root_for(pointer: Path, manifest: dict[str, Any]) -> Path:
    family = SCHEMA_TO_FAMILY[manifest_schema(manifest)]
    required_names = list(BASE_REQUIRED_SOURCE_ARTIFACTS)
    if family == "dgd_ablation":
        required_names.extend(["selection_decision", "validation_summary"])
    artifacts = manifest["artifacts"]
    relative_tests = [manifest["bundle_manifest"]]
    relative_tests.extend(artifacts[name]["path"] for name in required_names)
    candidates = [ARTIFACT_ROOT, PROJECT_ROOT / ".notebook" / "artifacts", *pointer.parents]
    seen: set[str] = set()
    for root in candidates:
        resolved_root = root.resolve()
        if str(resolved_root) in seen:
            continue
        seen.add(str(resolved_root))
        try:
            if all(resolve_inside(resolved_root, relative_path).is_file() for relative_path in relative_tests):
                return resolved_root
        except (OSError, ValueError):
            continue
    raise FileNotFoundError(f"Cannot resolve artifact root for {pointer}")


def load_json(path: Path) -> dict[str, Any]:
    value = strict_json_loads(path.read_text(encoding="utf-8"))
    if not isinstance(value, dict):
        raise ValueError(f"JSON document is not an object: {path}")
    return value


def read_csv_artifact(
    root: Path,
    generation_root: Path,
    manifest: dict[str, Any],
    name: str,
) -> tuple[pd.DataFrame, Path]:
    artifact = manifest["artifacts"][name]
    schema = manifest["output_schemas"][name]
    if (
        not isinstance(artifact, dict)
        or not isinstance(artifact.get("path"), str)
        or not isinstance(artifact.get("sha256"), str)
        or not isinstance(artifact.get("rows"), int)
        or artifact["rows"] < 0
    ):
        raise ValueError(f"Invalid artifact metadata for {name}")
    if (
        not isinstance(schema, dict)
        or not isinstance(schema.get("columns"), list)
        or not schema["columns"]
        or not isinstance(schema.get("read_csv_dtypes"), dict)
        or set(schema["read_csv_dtypes"]) != set(schema["columns"])
    ):
        raise ValueError(f"Invalid output schema for {name}")
    path = resolve_inside(root, artifact["path"])
    path.relative_to(generation_root)
    if not path.is_file() or sha256_file(path) != artifact["sha256"]:
        raise ValueError(f"Hash mismatch for artifact {name}: {path}")
    table = pd.read_csv(path, dtype=schema["read_csv_dtypes"])
    if list(table.columns) != schema["columns"] or len(table) != artifact["rows"]:
        raise ValueError(f"CSV contract mismatch for artifact {name}: {path}")
    return table, path


def ensure_finite_table(
    table: pd.DataFrame, name: str, required_numeric_columns: Iterable[str]
) -> None:
    if table.empty:
        raise ValueError(f"Artifact {name} is empty")
    missing = set(required_numeric_columns) - set(table.columns)
    if missing:
        raise ValueError(f"Artifact {name} is missing numeric columns: {sorted(missing)}")
    for column in required_numeric_columns:
        values = pd.to_numeric(table[column], errors="coerce").to_numpy(
            dtype=np.float64, na_value=np.nan
        )
        if not np.isfinite(values).all():
            raise ValueError(f"Artifact {name} has invalid numeric values in {column}")
    numeric_columns = [
        column
        for column in table.columns
        if pd.api.types.is_numeric_dtype(table[column].dtype)
    ]
    if not numeric_columns:
        raise ValueError(f"Artifact {name} has no numeric columns")
    for column in numeric_columns:
        values = table[column].to_numpy(dtype=np.float64, na_value=np.nan)
        if not np.isfinite(values).all():
            raise ValueError(f"Artifact {name} has non-finite values in {column}")


def validate_phase_rows(table: pd.DataFrame, name: str) -> None:
    if "phase" not in table.columns:
        raise ValueError(f"Artifact {name} has no phase column")
    counts = table["phase"].astype(str).value_counts().to_dict()
    expected = {phase: 1 for phase in PHASE_ORDER}
    if counts != expected:
        raise ValueError(f"Artifact {name} phase cardinality is {counts}, expected {expected}")


def false_value(value: Any) -> bool:
    if isinstance(value, str):
        return value.strip().lower() in {"false", "0"}
    return not bool(value)


def verify_source_run(pointer: Path, manifest: dict[str, Any]) -> dict[str, Any]:
    schema = manifest_schema(manifest)
    family = SCHEMA_TO_FAMILY[schema]
    source_status = manifest_status(manifest)
    if source_status not in {"PASS", "WARN"}:
        raise ValueError(f"Source status is not PASS/WARN: {source_status!r}")

    bundle_id = manifest.get("bundle_id")
    if not isinstance(bundle_id, str) or not bundle_id:
        raise ValueError("Source bundle_id must be a nonempty string")
    run_config = manifest.get("run_config")
    if not isinstance(run_config, dict) or run_config.get("schema_version") != schema:
        raise ValueError("Source run_config schema does not match the manifest schema")
    expected_run_hash = hashlib.sha256(
        strict_json_dumps(run_config, sort_keys=True, separators=(",", ":")).encode()
    ).hexdigest()
    if manifest.get("run_config_sha256") != expected_run_hash:
        raise ValueError("Source run_config hash mismatch")
    seed_value = run_config.get("seed")
    if isinstance(seed_value, bool) or not isinstance(seed_value, int):
        raise ValueError("Source run_config seed must be an integer")
    seed = seed_value
    created_at_utc = manifest.get("created_at_utc")
    if not isinstance(created_at_utc, str) or not created_at_utc:
        raise ValueError("Source created_at_utc must be a nonempty string")
    created_at = datetime.fromisoformat(created_at_utc.replace("Z", "+00:00"))
    if created_at.tzinfo is None:
        raise ValueError("Source created_at_utc must include a timezone")

    checks = manifest.get("checks")
    if not isinstance(checks, list) or not checks or not all(
        isinstance(row, dict) and row.get("status") == "PASS" for row in checks
    ):
        raise ValueError("Source structural checks are empty or not all PASS")
    quality_status = manifest.get("quality_status")
    if quality_status is not None:
        quality_checks = manifest.get("quality_checks")
        if (
            not isinstance(quality_checks, list)
            or not quality_checks
            or not all(
                isinstance(row, dict) and row.get("status") in {"PASS", "WARN"}
                for row in quality_checks
            )
        ):
            raise ValueError("Source quality status has no quality checks")
        expected_quality_status = (
            "WARN"
            if any(row["status"] == "WARN" for row in quality_checks)
            else "PASS"
        )
        if quality_status != expected_quality_status or source_status != expected_quality_status:
            raise ValueError("Source quality status is inconsistent with its quality checks")

    upstream = manifest.get("upstream_protocol")
    if (
        not isinstance(upstream, dict)
        or upstream.get("schema_version") != EXPECTED_PROTOCOL_SCHEMA
        or not isinstance(upstream.get("pointer_sha256"), str)
        or not upstream["pointer_sha256"]
    ):
        raise ValueError("Source upstream protocol identity is invalid")

    artifacts = manifest.get("artifacts")
    output_schemas = manifest.get("output_schemas")
    required_names = list(BASE_REQUIRED_SOURCE_ARTIFACTS)
    if family == "dgd_ablation":
        required_names.extend(["selection_decision", "validation_summary"])
    if not isinstance(artifacts, dict) or not set(required_names) <= set(artifacts):
        raise ValueError(f"Source is missing required artifacts: {required_names}")
    if not isinstance(output_schemas, dict) or not set(required_names) <= set(output_schemas):
        raise ValueError(f"Source is missing required output schemas: {required_names}")

    root = artifact_root_for(pointer, manifest)
    bundle_manifest_path = resolve_inside(root, manifest["bundle_manifest"])
    generation_root = bundle_manifest_path.parent
    if (
        bundle_manifest_path.name != "manifest.json"
        or generation_root.name != bundle_id
        or generation_root.parent.name != "generations"
    ):
        raise ValueError("Source bundle_manifest is not a canonical immutable generation path")
    if bundle_manifest_path.read_bytes() != pointer.read_bytes():
        raise ValueError("Source pointer and immutable generation manifest differ")
    if load_json(bundle_manifest_path) != manifest:
        raise ValueError("Source immutable manifest content mismatch")

    loaded: dict[str, pd.DataFrame] = {}
    artifact_paths: dict[str, Path] = {}
    for name in required_names:
        loaded[name], artifact_paths[name] = read_csv_artifact(
            root, generation_root, manifest, name
        )

    metrics = loaded["evaluation_metrics"]
    thresholds = loaded["validation_thresholds"]
    predictions = loaded["evaluation_predictions"]
    ensure_finite_table(metrics, "evaluation_metrics", ["threshold", *METRIC_COLUMNS])
    ensure_finite_table(thresholds, "validation_thresholds", ["threshold"])
    ensure_finite_table(predictions, "evaluation_predictions", ["score"])

    if family == "dgd_ablation":
        selected_variant = str(manifest.get("summary", {}).get("advance_to_multiseed", ""))
        if not selected_variant:
            raise ValueError("Ablation source does not declare advance_to_multiseed")
        decision = loaded["selection_decision"]
        validation_summary = loaded["validation_summary"]
        ensure_finite_table(
            validation_summary,
            "validation_summary",
            ["mean_validation_f1", "mean_validation_auc"],
        )
        if (
            len(decision) != 1
            or str(decision.iloc[0].get("advance_to_multiseed", "")) != selected_variant
            or str(decision.iloc[0].get("selection_metric", ""))
            != "mean_validation_f1"
            or not false_value(decision.iloc[0].get("uses_evaluation_for_selection", True))
        ):
            raise ValueError("Ablation selection decision is missing, inconsistent, or uses evaluation")
        selected_rows = validation_summary[
            validation_summary["variant"].astype(str).eq(selected_variant)
        ]
        best_f1 = float(validation_summary["mean_validation_f1"].max())
        f1_tied = validation_summary[
            np.isclose(
                validation_summary["mean_validation_f1"].astype(float),
                best_f1,
                rtol=0.0,
                atol=1e-12,
            )
        ]
        best_tied_auc = float(f1_tied["mean_validation_auc"].max())
        if (
            len(selected_rows) != 1
            or not np.isclose(
                float(selected_rows.iloc[0]["mean_validation_f1"]),
                best_f1,
                rtol=0.0,
                atol=1e-12,
            )
            or not np.isclose(
                float(selected_rows.iloc[0]["mean_validation_auc"]),
                best_tied_auc,
                rtol=0.0,
                atol=1e-12,
            )
        ):
            raise ValueError("Ablation selected variant does not match validation summary")
        if "variant" not in metrics or "variant" not in thresholds or "variant" not in predictions:
            raise ValueError("Ablation artifacts do not expose variant scope")
        metrics = metrics[metrics["variant"].astype(str).eq(selected_variant)].copy()
        thresholds = thresholds[thresholds["variant"].astype(str).eq(selected_variant)].copy()
        prediction_variants = set(predictions["variant"].astype(str))
        if selected_variant not in prediction_variants:
            raise ValueError("Ablation prediction artifact omits the selected variant")
        prediction_scope = (
            "all_variants" if len(prediction_variants) > 1 else "selected_variant_only"
        )
        model_label = "DGD-Ablation:selected"
    else:
        selected_variant = ""
        expected_label = EXPECTED_FAMILIES[family]["label"]
        if manifest.get("model_name") != expected_label:
            raise ValueError(
                f"Source model_name is {manifest.get('model_name')!r}, expected {expected_label!r}"
            )
        model_label = expected_label
        prediction_scope = "active_model"
        metrics = metrics.copy()
        thresholds = thresholds.copy()
        if "variant" not in metrics.columns:
            metrics["variant"] = ""
        if "variant" not in thresholds.columns:
            thresholds["variant"] = ""

    validate_phase_rows(metrics, "evaluation_metrics")
    validate_phase_rows(thresholds, "validation_thresholds")
    if "split" in metrics and set(metrics["split"].astype(str)) != {"evaluation"}:
        raise ValueError("Evaluation metrics contain a non-evaluation split")
    if "split" in thresholds and set(thresholds["split"].astype(str)) != {"validation"}:
        raise ValueError("Validation thresholds contain a non-validation split")
    metric_thresholds = metrics.set_index("phase")["threshold"].astype(float)
    validation_thresholds = thresholds.set_index("phase")["threshold"].astype(float)
    for phase in PHASE_ORDER:
        if not np.isclose(
            metric_thresholds[phase], validation_thresholds[phase], rtol=0.0, atol=1e-12
        ):
            raise ValueError(
                f"Evaluation threshold for {phase} does not match its validation threshold"
            )

    return {
        "family": family,
        "model_label": model_label,
        "selected_variant": selected_variant,
        "prediction_scope": prediction_scope,
        "source_status": source_status,
        "schema_version": schema,
        "seed": seed,
        "bundle_id": bundle_id,
        "created_at_utc": created_at_utc,
        "protocol_schema_version": upstream["schema_version"],
        "protocol_pointer_sha256": upstream["pointer_sha256"],
        "manifest_path": str(bundle_manifest_path),
        "artifact_root": str(root),
        "manifest": manifest,
        "metrics": metrics,
        "thresholds": thresholds,
        "artifact_paths": artifact_paths,
    }


DISCOVERED_RUNS: list[dict[str, Any]] = []
DISCOVERY_ERRORS: list[str] = []
verified_bundle_digests: dict[tuple[str, str], str] = {}
for pointer in candidate_manifest_paths():
    try:
        manifest = load_json(pointer)
        schema = manifest_schema(manifest)
        if schema not in SCHEMA_TO_FAMILY:
            continue
        bundle_id = manifest.get("bundle_id")
        if not isinstance(bundle_id, str) or not bundle_id:
            raise ValueError("Source bundle_id must be a nonempty string")
        key = (schema, bundle_id)
        pointer_digest = sha256_file(pointer)
        if key in verified_bundle_digests:
            if verified_bundle_digests[key] != pointer_digest:
                DISCOVERY_ERRORS.append(
                    f"{pointer}: conflicting manifest content for verified bundle {bundle_id}"
                )
            continue
        DISCOVERED_RUNS.append(verify_source_run(pointer, manifest))
        verified_bundle_digests[key] = pointer_digest
    except Exception as error:
        DISCOVERY_ERRORS.append(f"{pointer}: {type(error).__name__}: {error}")

if not DISCOVERED_RUNS:
    raise RuntimeError("No verified model result manifests found for notebooks 03-06")

INTERNAL_RUN_FIELDS = {"manifest", "metrics", "thresholds", "artifact_paths"}


def run_registry_record(run: dict[str, Any]) -> dict[str, Any]:
    return {key: value for key, value in run.items() if key not in INTERNAL_RUN_FIELDS}


latest_runs: dict[tuple[str, str, int], dict[str, Any]] = {}
for run in sorted(
    DISCOVERED_RUNS,
    key=lambda row: (
        row["family"],
        row["model_label"],
        row["seed"],
        row["created_at_utc"],
        row["bundle_id"],
    ),
):
    latest_runs[(run["family"], run["model_label"], run["seed"])] = run

LATEST_RUNS = list(latest_runs.values())
RUN_REGISTRY_ALL = pd.DataFrame([run_registry_record(run) for run in LATEST_RUNS])
RUN_REGISTRY_ALL = RUN_REGISTRY_ALL.sort_values(
    ["family", "seed", "created_at_utc", "bundle_id"]
).reset_index(drop=True)
ACTIVE_RUNS = sorted(
    [run for run in LATEST_RUNS if run["seed"] in TARGET_SEED_SET],
    key=lambda row: (row["family"], row["model_label"], row["seed"]),
)
RUN_REGISTRY = pd.DataFrame([run_registry_record(run) for run in ACTIVE_RUNS])
if RUN_REGISTRY.empty:
    raise RuntimeError("No verified source bundles match the requested target seeds")
RUN_REGISTRY = RUN_REGISTRY.sort_values(["family", "seed"]).reset_index(drop=True)
NON_TARGET_RUNS = RUN_REGISTRY_ALL.loc[
    ~RUN_REGISTRY_ALL["seed"].astype(int).isin(TARGET_SEED_SET)
].reset_index(drop=True)

display(RUN_REGISTRY[["family", "model_label", "selected_variant", "seed", "source_status", "bundle_id", "created_at_utc"]])
if not NON_TARGET_RUNS.empty:
    display(NON_TARGET_RUNS[["family", "model_label", "seed", "source_status", "bundle_id"]])
if DISCOVERY_ERRORS:
    show_records([{"discovery_warning": error} for error in DISCOVERY_ERRORS[:10]])

,family,model_label,selected_variant,seed,source_status,bundle_id,created_at_utc
0,dgd,DGD,,2025,PASS,20260716T224624-s2025-da86c7bb3b3d,2026-07-16T22:49:49.869599+00:00
1,dgd,DGD,,3407,PASS,20260716T225641-s3407-ad2318bf266f,2026-07-16T23:00:03.742725+00:00
2,dgd,DGD,,4517,PASS,20260716T230004-s4517-f251f88018dd,2026-07-16T23:03:23.593273+00:00
3,dgd,DGD,,7788,PASS,20260716T224950-s7788-641c79caa5d4,2026-07-16T22:53:12.494885+00:00
4,dgd,DGD,,9999,PASS,20260716T225312-s9999-84401d95a687,2026-07-16T22:56:41.624334+00:00
5,dgd_ablation,DGD-Ablation:selected,full_dgd,2025,PASS,20260717T021403351014-s2025-cc8981404b56,2026-07-17T02:14:06.815011+00:00
6,dgd_ablation,DGD-Ablation:selected,full_dgd,3407,PASS,20260717T023114264309-s3407-6803fb024589,2026-07-17T02:31:17.915260+00:00
7,dgd_ablation,DGD-Ablation:selected,full_dgd,4517,PASS,20260717T023658001768-s4517-5cf69cee9acf,2026-07-17T02:37:01.487380+00:00
8,dgd_ablation,DGD-Ablation:selected,full_dgd,7788,PASS,20260717T021946974431-s7788-001c9cfe9c5b,2026-07-17T02:19:50.655282+00:00
9,dgd_ablation,DGD-Ablation:selected,full_dgd,9999,PASS,20260717T022529670475-s9999-7c3011e765ac,2026-07-17T02:25:33.121799+00:00


In [3]:
metric_parts: list[pd.DataFrame] = []
threshold_parts: list[pd.DataFrame] = []
prediction_registry_rows: list[dict[str, Any]] = []
artifact_audit_rows: list[dict[str, Any]] = []

for run in ACTIVE_RUNS:
    manifest = run["manifest"]
    metrics = run["metrics"].copy()
    thresholds = run["thresholds"].copy()
    prediction_artifact = manifest["artifacts"]["evaluation_predictions"]
    prediction_path = str(run["artifact_paths"]["evaluation_predictions"])
    prediction_sha = prediction_artifact["sha256"]

    for table in (metrics, thresholds):
        table.insert(0, "family", run["family"])
        table.insert(1, "model_label", run["model_label"])
        table.insert(2, "seed", run["seed"])
        table.insert(3, "bundle_id", run["bundle_id"])
        table.insert(4, "schema_version", run["schema_version"])
        table.insert(5, "protocol_pointer_sha256", run["protocol_pointer_sha256"])
        table.insert(6, "source_status", run["source_status"])

    metric_parts.append(metrics)
    threshold_parts.append(thresholds)
    prediction_registry_rows.append(
        {
            "family": run["family"],
            "model_label": run["model_label"],
            "seed": run["seed"],
            "bundle_id": run["bundle_id"],
            "selected_variant": run["selected_variant"],
            "prediction_scope": run["prediction_scope"],
            "source_status": run["source_status"],
            "evaluation_predictions_path": prediction_path,
            "evaluation_predictions_sha256": prediction_sha,
        }
    )
    for name in ("evaluation_metrics", "validation_thresholds", "evaluation_predictions"):
        artifact = manifest.get("artifacts", {}).get(name)
        artifact_audit_rows.append(
            {
                "family": run["family"],
                "model_label": run["model_label"],
                "seed": run["seed"],
                "bundle_id": run["bundle_id"],
                "source_status": run["source_status"],
                "artifact": name,
                "present": True,
                "verified": True,
                "rows": artifact["rows"],
                "sha256": artifact["sha256"],
            }
        )

PER_RUN_METRICS = pd.concat(metric_parts, ignore_index=True)
VALIDATION_THRESHOLDS = pd.concat(threshold_parts, ignore_index=True)
PREDICTION_ARTIFACT_REGISTRY = pd.DataFrame(prediction_registry_rows)
ARTIFACT_AUDIT = pd.DataFrame(artifact_audit_rows)

if set(PER_RUN_METRICS["phase"].astype(str)) != set(PHASE_ORDER):
    raise RuntimeError("Per-run metrics do not exactly cover the required phases")
run_phase_columns = ["family", "model_label", "seed", "phase"]
if PER_RUN_METRICS.duplicated(run_phase_columns).any():
    raise RuntimeError("Duplicate active metric rows would bias multi-seed aggregation")
if VALIDATION_THRESHOLDS.duplicated(run_phase_columns).any():
    raise RuntimeError("Duplicate active validation-threshold rows are not allowed")
if not set(PER_RUN_METRICS["seed"].astype(int)) <= TARGET_SEED_SET:
    raise RuntimeError("Non-target seeds reached active metric aggregation")

display(PER_RUN_METRICS[["model_label", "seed", "phase", "f1", "roc_auc"]].sort_values(["model_label", "seed", "phase"]))

,model_label,seed,phase,f1,roc_auc
0,DGD,2025,Cold,0.612464,0.732979
1,DGD,2025,Warm A,0.670894,0.782171
2,DGD,2025,Warm B,0.688299,0.796279
3,DGD,2025,Warm C,0.690961,0.801468
4,DGD,3407,Cold,0.614018,0.733348
...,...,...,...,...,...
75,LightGCN,7788,Warm C,0.579760,0.483060
76,LightGCN,9999,Cold,0.579778,0.484254
77,LightGCN,9999,Warm A,0.579778,0.476783
78,LightGCN,9999,Warm B,0.579760,0.474103


In [4]:
available_metric_columns = [column for column in METRIC_COLUMNS if column in PER_RUN_METRICS.columns]

aggregate_rows: list[dict[str, Any]] = []
for (family, model_label, phase), group in PER_RUN_METRICS.groupby(["family", "model_label", "phase"], observed=True):
    observed_seeds = set(group["seed"].astype(int))
    row: dict[str, Any] = {
        "family": family,
        "model_label": model_label,
        "phase": phase,
        "seed_count": int(group["seed"].nunique()),
        "target_seed_count_observed": len(observed_seeds & TARGET_SEED_SET),
        "target_seed_count": len(TARGET_SEEDS),
        "target_seed_count_met": bool(TARGET_SEED_SET <= observed_seeds),
    }
    for metric in available_metric_columns:
        values = pd.to_numeric(group[metric], errors="coerce")
        row[f"{metric}_mean"] = float(values.mean())
        row[f"{metric}_std"] = float(values.std(ddof=1)) if values.count() > 1 else np.nan
        row[f"{metric}_sem"] = float(values.sem(ddof=1)) if values.count() > 1 else np.nan
    aggregate_rows.append(row)

AGGREGATE_METRICS = pd.DataFrame(aggregate_rows)
phase_rank_rows: list[dict[str, Any]] = []
for phase, group in AGGREGATE_METRICS.groupby("phase", observed=True):
    ranked = group.sort_values(["f1_mean", "roc_auc_mean"], ascending=False).reset_index(drop=True)
    for rank, record in enumerate(ranked.to_dict(orient="records"), start=1):
        phase_rank_rows.append({"phase": phase, "rank": rank, **record})
PHASE_RANKING = pd.DataFrame(phase_rank_rows)
OVERALL_RANKING = (
    AGGREGATE_METRICS.groupby(["family", "model_label"], observed=True)
    .agg(
        phases=("phase", "nunique"),
        seed_count_min=("seed_count", "min"),
        f1_mean=("f1_mean", "mean"),
        roc_auc_mean=("roc_auc_mean", "mean"),
    )
    .reset_index()
    .sort_values(["f1_mean", "roc_auc_mean"], ascending=False)
    .reset_index(drop=True)
)
OVERALL_RANKING.insert(0, "rank", np.arange(1, len(OVERALL_RANKING) + 1))

display(AGGREGATE_METRICS.sort_values(["model_label", "phase"]))
display(OVERALL_RANKING)

,family,model_label,phase,seed_count,target_seed_count_observed,target_seed_count,target_seed_count_met,accuracy_mean,accuracy_std,accuracy_sem,...,recall_sem,f1_mean,f1_std,f1_sem,roc_auc_mean,roc_auc_std,roc_auc_sem,predicted_positive_rate_mean,predicted_positive_rate_std,predicted_positive_rate_sem
0,dgd,DGD,Cold,5,5,5,True,0.502986,0.008688,0.003885,...,0.002663,0.613022,0.002738,0.001224,0.731973,0.002168,0.000970,0.876048,0.013463,0.006021
1,dgd,DGD,Warm A,5,5,5,True,0.650506,0.005891,0.002635,...,0.003016,0.668832,0.002370,0.001060,0.781407,0.002523,0.001128,0.647060,0.011030,0.004933
2,dgd,DGD,Warm B,5,5,5,True,0.701676,0.004410,0.001972,...,0.003931,0.687744,0.000912,0.000408,0.796202,0.001574,0.000704,0.547135,0.011563,0.005171
3,dgd,DGD,Warm C,5,5,5,True,0.712470,0.003087,0.001381,...,0.004092,0.692112,0.001231,0.000551,0.801693,0.001780,0.000796,0.525661,0.010267,0.004592
4,dgd_ablation,DGD-Ablation:selected,Cold,5,5,5,True,0.511619,0.002273,0.001016,...,0.001116,0.615612,0.000791,0.000354,0.732364,0.002165,0.000968,0.862315,0.003985,0.001782
5,dgd_ablation,DGD-Ablation:selected,Warm A,5,5,5,True,0.609505,0.008601,0.003846,...,0.003724,0.650755,0.003055,0.001366,0.766333,0.001253,0.000560,0.709788,0.015235,0.006813
6,dgd_ablation,DGD-Ablation:selected,Warm B,5,5,5,True,0.668176,0.007767,0.003474,...,0.005656,0.674237,0.002157,0.000965,0.784857,0.001213,0.000542,0.610304,0.017945,0.008025
7,dgd_ablation,DGD-Ablation:selected,Warm C,5,5,5,True,0.692501,0.008848,0.003957,...,0.007935,0.685120,0.001859,0.000831,0.795755,0.001078,0.000482,0.568247,0.023229,0.010388
8,emerg,EmerG,Cold,5,5,5,True,0.500245,0.007985,0.003571,...,0.002166,0.612787,0.002750,0.001230,0.738041,0.005545,0.002480,0.882360,0.011738,0.005249
9,emerg,EmerG,Warm A,5,5,5,True,0.663975,0.007416,0.003316,...,0.005611,0.678676,0.002492,0.001115,0.793040,0.002476,0.001107,0.637461,0.017176,0.007681


,rank,family,model_label,phases,seed_count_min,f1_mean,roc_auc_mean
0,1,emerg,EmerG,4,5,0.669042,0.783981
1,2,dgd,DGD,4,5,0.665427,0.777819
2,3,dgd_ablation,DGD-Ablation:selected,4,5,0.656431,0.769827
3,4,lightgcn,LightGCN,4,5,0.579754,0.491769


In [5]:
AUDIT_ROWS: list[dict[str, Any]] = []


def audit(name: str, condition: bool, observed: Any, expected: Any, severity: str = "ERROR") -> None:
    AUDIT_ROWS.append(
        {
            "check": name,
            "status": "PASS" if condition else severity,
            "observed": observed,
            "expected": expected,
            "severity": severity,
        }
    )


present_families = set(RUN_REGISTRY["family"])
protocol_schemas = sorted(set(RUN_REGISTRY["protocol_schema_version"].astype(str)))
protocol_hashes = sorted(set(RUN_REGISTRY["protocol_pointer_sha256"].astype(str)))
source_warning_runs = RUN_REGISTRY.loc[
    RUN_REGISTRY["source_status"].eq("WARN"),
    ["family", "model_label", "seed", "bundle_id"],
].to_dict(orient="records")
selected_ablation_variants = sorted(
    variant
    for variant in RUN_REGISTRY.loc[
        RUN_REGISTRY["family"].eq("dgd_ablation"), "selected_variant"
    ].astype(str).unique()
    if variant
)
expected_model_labels = {
    family: (
        "DGD-Ablation:selected"
        if family == "dgd_ablation"
        else contract["label"]
    )
    for family, contract in EXPECTED_FAMILIES.items()
}
seed_coverage_rows: list[dict[str, Any]] = []
for family, model_label in expected_model_labels.items():
    observed_seeds = set(
        RUN_REGISTRY.loc[RUN_REGISTRY["family"].eq(family), "seed"].astype(int)
    )
    for seed in TARGET_SEEDS:
        seed_coverage_rows.append(
            {
                "family": family,
                "model_label": model_label,
                "seed": seed,
                "expected_seed": True,
                "present": seed in observed_seeds,
            }
        )
    non_target_seeds = set(
        NON_TARGET_RUNS.loc[NON_TARGET_RUNS["family"].eq(family), "seed"].astype(int)
    )
    for seed in sorted(non_target_seeds):
        seed_coverage_rows.append(
            {
                "family": family,
                "model_label": model_label,
                "seed": seed,
                "expected_seed": False,
                "present": True,
            }
        )
SEED_COVERAGE = pd.DataFrame(seed_coverage_rows)
missing_seed_map = (
    SEED_COVERAGE.loc[SEED_COVERAGE["expected_seed"] & ~SEED_COVERAGE["present"]]
    .groupby("model_label", observed=True)["seed"]
    .apply(lambda values: sorted(int(value) for value in values))
    .to_dict()
)
expected_run_keys = {
    (family, model_label, seed)
    for family, model_label in expected_model_labels.items()
    for seed in TARGET_SEEDS
}
actual_run_keys = set(
    zip(RUN_REGISTRY["family"], RUN_REGISTRY["model_label"], RUN_REGISTRY["seed"])
)
expected_run_phase_keys = {
    (*run_key, phase) for run_key in expected_run_keys for phase in PHASE_ORDER
}
metric_run_phase_keys = set(
    zip(
        PER_RUN_METRICS["family"],
        PER_RUN_METRICS["model_label"],
        PER_RUN_METRICS["seed"],
        PER_RUN_METRICS["phase"],
    )
)
threshold_run_phase_keys = set(
    zip(
        VALIDATION_THRESHOLDS["family"],
        VALIDATION_THRESHOLDS["model_label"],
        VALIDATION_THRESHOLDS["seed"],
        VALIDATION_THRESHOLDS["phase"],
    )
)
target_seed_complete = bool(
    actual_run_keys == expected_run_keys
    and metric_run_phase_keys == expected_run_phase_keys
    and threshold_run_phase_keys == expected_run_phase_keys
)

audit("required model families present", set(EXPECTED_FAMILIES) <= present_families, sorted(present_families), sorted(EXPECTED_FAMILIES))
audit("protocol schema", protocol_schemas == [EXPECTED_PROTOCOL_SCHEMA], protocol_schemas, [EXPECTED_PROTOCOL_SCHEMA])
audit("single protocol identity", len(protocol_hashes) == 1 and bool(protocol_hashes[0]), protocol_hashes, "one nonempty protocol pointer sha256")
audit("phase coverage", set(PER_RUN_METRICS["phase"].astype(str)) == set(PHASE_ORDER), sorted(PER_RUN_METRICS["phase"].unique()), PHASE_ORDER)
audit("metric rows unique", len(metric_run_phase_keys) == len(PER_RUN_METRICS), len(PER_RUN_METRICS), len(metric_run_phase_keys))
audit("threshold rows unique", len(threshold_run_phase_keys) == len(VALIDATION_THRESHOLDS), len(VALIDATION_THRESHOLDS), len(threshold_run_phase_keys))
audit("target run-phase coverage", target_seed_complete, {"runs": len(actual_run_keys), "metric_rows": len(metric_run_phase_keys), "threshold_rows": len(threshold_run_phase_keys)}, {"runs": len(expected_run_keys), "metric_rows": len(expected_run_phase_keys), "threshold_rows": len(expected_run_phase_keys)}, severity="WARN")
audit("non-target seeds excluded from metrics", set(PER_RUN_METRICS["seed"].astype(int)) <= TARGET_SEED_SET, sorted(set(PER_RUN_METRICS["seed"].astype(int)) - TARGET_SEED_SET), [])
audit("prediction artifacts verified", PREDICTION_ARTIFACT_REGISTRY["evaluation_predictions_path"].notna().all(), int(PREDICTION_ARTIFACT_REGISTRY["evaluation_predictions_path"].notna().sum()), len(PREDICTION_ARTIFACT_REGISTRY))
audit("source model quality clean", not source_warning_runs, source_warning_runs, "no source WARN runs", severity="WARN")
audit("selected ablation stable", len(selected_ablation_variants) <= 1, selected_ablation_variants, "one validation-selected ablation variant across seeds", severity="WARN")
audit("target seeds complete", target_seed_complete, missing_seed_map, "no missing target runs or phases", severity="WARN")
audit("manifest discovery clean", not DISCOVERY_ERRORS, DISCOVERY_ERRORS[:10], "no discovery errors", severity="WARN")

REPRO_AUDIT = pd.DataFrame(AUDIT_ROWS)
EVALUATION_PASS = not REPRO_AUDIT["status"].eq("ERROR").any()
EVALUATION_STATUS = "WARN" if REPRO_AUDIT["status"].eq("WARN").any() else "PASS"
display(REPRO_AUDIT)
if not EVALUATION_PASS:
    raise RuntimeError("Blocking multi-seed evaluation audit checks failed")

,check,status,observed,expected,severity
0,required model families present,PASS,"[dgd, dgd_ablation, emerg, lightgcn]","[dgd, dgd_ablation, emerg, lightgcn]",ERROR
1,protocol schema,PASS,[ml1m-coldstart-v1],[ml1m-coldstart-v1],ERROR
2,single protocol identity,PASS,[32555f37de275f3496da33cf8c1721a440cb8eb0a32b8...,one nonempty protocol pointer sha256,ERROR
3,phase coverage,PASS,"[Cold, Warm A, Warm B, Warm C]","[Cold, Warm A, Warm B, Warm C]",ERROR
4,metric rows unique,PASS,80,80,ERROR
5,threshold rows unique,PASS,80,80,ERROR
6,target run-phase coverage,PASS,"{'runs': 20, 'metric_rows': 80, 'threshold_row...","{'runs': 20, 'metric_rows': 80, 'threshold_row...",WARN
7,non-target seeds excluded from metrics,PASS,[],[],ERROR
8,prediction artifacts verified,PASS,20,20,ERROR
9,source model quality clean,WARN,"[{'family': 'lightgcn', 'model_label': 'LightG...",no source WARN runs,WARN


In [6]:
def json_ready(value: Any) -> Any:
    if value is None or isinstance(value, (str, int, bool)):
        return value
    if isinstance(value, float):
        if not np.isfinite(value):
            raise ValueError(f"Non-finite value cannot be serialized to JSON: {value}")
        return value
    if isinstance(value, dict):
        return {str(key): json_ready(item) for key, item in value.items()}
    if isinstance(value, (list, tuple, set)):
        return [json_ready(item) for item in value]
    if hasattr(value, "item"):
        return json_ready(value.item())
    return str(value)


def write_text(path: Path, content: str) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(f".{path.name}.{uuid.uuid4().hex}.tmp")
    temporary.write_text(content, encoding="utf-8")
    temporary.replace(path)


def write_csv(path: Path, table: pd.DataFrame) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(f".{path.name}.{uuid.uuid4().hex}.tmp")
    table.to_csv(temporary, index=False, lineterminator="\n")
    temporary.replace(path)


def relative_output(path: Path) -> str:
    return str(path.resolve().relative_to(ARTIFACT_ROOT.resolve()))


def csv_schema(table: pd.DataFrame) -> dict[str, Any]:
    def dtype_name(dtype: Any) -> str:
        name = str(dtype)
        return "string" if name in {"str", "string"} or name.startswith("string") else name

    return {
        "columns": list(table.columns),
        "read_csv_dtypes": {column: dtype_name(dtype) for column, dtype in table.dtypes.items()},
    }


bundle_id = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S") + "-" + uuid.uuid4().hex[:12]
staging_root = OUTPUT_ROOT / f".staging-{bundle_id}"
generation_root = OUTPUT_ROOT / "generations" / bundle_id
pointer_path = OUTPUT_ROOT / "manifest.json"
previous_pointer_text = pointer_path.read_text(encoding="utf-8") if pointer_path.is_file() else None
generation_published = False
pointer_write_attempted = False
staging_root.mkdir(parents=True, exist_ok=False)

OUTPUT_TABLES = {
    "run_registry": RUN_REGISTRY.drop(columns=["artifact_root"], errors="ignore"),
    "non_target_runs": NON_TARGET_RUNS.drop(columns=["artifact_root"], errors="ignore"),
    "per_run_metrics": PER_RUN_METRICS,
    "validation_thresholds": VALIDATION_THRESHOLDS,
    "aggregate_metrics": AGGREGATE_METRICS,
    "phase_ranking": PHASE_RANKING,
    "overall_ranking": OVERALL_RANKING,
    "seed_coverage": SEED_COVERAGE,
    "prediction_artifact_registry": PREDICTION_ARTIFACT_REGISTRY,
    "artifact_audit": ARTIFACT_AUDIT,
    "reproducibility_audit": REPRO_AUDIT,
}
OUTPUT_ARTIFACTS: dict[str, Any] = {}

try:
    for name, table in OUTPUT_TABLES.items():
        staging_path = staging_root / f"{name}.csv"
        published_path = generation_root / f"{name}.csv"
        write_csv(staging_path, table)
        OUTPUT_ARTIFACTS[name] = {"path": relative_output(published_path), "sha256": sha256_file(staging_path), "rows": len(table)}

    manifest = {
        "evaluation_schema_version": RUN_CONFIG["schema_version"],
        "evaluation_status": EVALUATION_STATUS,
        "coverage_status": "COMPLETE" if target_seed_complete else "PARTIAL",
        "bundle_id": bundle_id,
        "bundle_manifest": relative_output(generation_root / "manifest.json"),
        "created_at_utc": datetime.now(timezone.utc).isoformat(),
        "run_config": RUN_CONFIG,
        "run_config_sha256": RUN_CONFIG_SHA256,
        "upstream_protocol_pointer_sha256": protocol_hashes[0] if protocol_hashes else "",
        "summary": {
            "models": int(RUN_REGISTRY["model_label"].nunique()),
            "runs": int(len(RUN_REGISTRY)),
            "non_target_runs_recorded": int(len(NON_TARGET_RUNS)),
            "target_seed_count": len(TARGET_SEEDS),
            "target_seed_complete": bool(target_seed_complete),
            "missing_target_seeds": missing_seed_map,
            "source_warning_runs": source_warning_runs,
            "selected_ablation_variants": selected_ablation_variants,
            "best_model_by_mean_f1": str(OVERALL_RANKING.iloc[0]["model_label"]),
            "best_mean_f1": float(OVERALL_RANKING.iloc[0]["f1_mean"]),
            "best_mean_auc": float(OVERALL_RANKING.iloc[0]["roc_auc_mean"]),
        },
        "checks": REPRO_AUDIT.to_dict(orient="records"),
        "artifacts": OUTPUT_ARTIFACTS,
        "output_schemas": {name: csv_schema(table) for name, table in OUTPUT_TABLES.items()},
        "training_contract": {
            "source": "verified notebooks 03-06 result bundles",
            "thresholds": "loaded from validation-selected thresholds in each source bundle",
            "evaluation": "held-out evaluation metrics only; no threshold or model selection performed here",
        },
    }

    manifest_text = strict_json_dumps(json_ready(manifest), indent=2, sort_keys=True) + "\n"
    write_text(staging_root / "manifest.json", manifest_text)
    generation_root.parent.mkdir(parents=True, exist_ok=True)
    staging_root.replace(generation_root)
    generation_published = True
    generation_hashes_match = all(
        sha256_file(resolve_inside(ARTIFACT_ROOT, artifact["path"])) == artifact["sha256"]
        for artifact in OUTPUT_ARTIFACTS.values()
    )
    if not generation_hashes_match or (generation_root / "manifest.json").read_text(encoding="utf-8") != manifest_text:
        raise RuntimeError("Published multi-seed generation failed pre-pointer verification")
    pointer_write_attempted = True
    write_text(pointer_path, manifest_text)
    if pointer_path.read_text(encoding="utf-8") != manifest_text:
        raise RuntimeError("Multi-seed pointer does not match the verified generation")
except Exception:
    try:
        if pointer_write_attempted:
            if previous_pointer_text is None:
                pointer_path.unlink(missing_ok=True)
            else:
                write_text(pointer_path, previous_pointer_text)
    finally:
        if staging_root.exists():
            shutil.rmtree(staging_root, ignore_errors=True)
        if generation_published and generation_root.exists():
            shutil.rmtree(generation_root)
    raise

MULTISEED_MANIFEST = manifest
MULTISEED_POINTER = pointer_path
show_records(
    [
        {
            "evaluation_status": manifest["evaluation_status"],
            "coverage_status": manifest["coverage_status"],
            "bundle_id": bundle_id,
            "manifest": str(pointer_path),
            "artifacts": len(OUTPUT_ARTIFACTS),
        }
    ]
)

,evaluation_status,coverage_status,bundle_id,manifest,artifacts
0,WARN,COMPLETE,20260717T024539-abd7b3ad0f9c,/kaggle/working/artifacts/evaluations/ml-1m/mu...,11


In [7]:
FINAL_CHECKS: list[dict[str, Any]] = list(REPRO_AUDIT.to_dict(orient="records"))


def final_check(name: str, condition: bool, observed: Any, expected: Any) -> None:
    FINAL_CHECKS.append(
        {"check": name, "status": "PASS" if condition else "ERROR", "observed": observed, "expected": expected, "severity": "ERROR"}
    )


manifest_pointer_matches = MULTISEED_POINTER.read_text(encoding="utf-8") == (generation_root / "manifest.json").read_text(encoding="utf-8")
artifact_hashes_match = all(
    sha256_file(resolve_inside(ARTIFACT_ROOT, artifact["path"])) == artifact["sha256"]
    for artifact in MULTISEED_MANIFEST["artifacts"].values()
)
final_check("manifest pointer matches bundle", manifest_pointer_matches, manifest_pointer_matches, True)
final_check("artifact hashes verify", artifact_hashes_match, artifact_hashes_match, True)
final_check("aggregate metrics exported", "aggregate_metrics" in MULTISEED_MANIFEST["artifacts"], True, True)
final_check("prediction registry exported", "prediction_artifact_registry" in MULTISEED_MANIFEST["artifacts"], True, True)

FINAL_AUDIT = pd.DataFrame(FINAL_CHECKS)
FINAL_PASS = not FINAL_AUDIT["status"].eq("ERROR").any()
NOTEBOOK_STATUS = (
    "BLOCKED"
    if not FINAL_PASS
    else "WARN"
    if MULTISEED_MANIFEST["evaluation_status"] == "WARN"
    else "READY"
)
display(FINAL_AUDIT)
display(Markdown("### Notebook 07 multi-seed evaluation: " + NOTEBOOK_STATUS))

if not FINAL_PASS:
    raise RuntimeError("Final multi-seed export checks failed")

display(
    Markdown(
        "**Next notebook:** notebook 08 should read this multiseed manifest and build the final tables/figures. "
        "If coverage is PARTIAL, rerun notebooks 03-06 with the missing seeds and rerun notebook 07."
    )
)

,check,status,observed,expected,severity
0,required model families present,PASS,"[dgd, dgd_ablation, emerg, lightgcn]","[dgd, dgd_ablation, emerg, lightgcn]",ERROR
1,protocol schema,PASS,[ml1m-coldstart-v1],[ml1m-coldstart-v1],ERROR
2,single protocol identity,PASS,[32555f37de275f3496da33cf8c1721a440cb8eb0a32b8...,one nonempty protocol pointer sha256,ERROR
3,phase coverage,PASS,"[Cold, Warm A, Warm B, Warm C]","[Cold, Warm A, Warm B, Warm C]",ERROR
4,metric rows unique,PASS,80,80,ERROR
5,threshold rows unique,PASS,80,80,ERROR
6,target run-phase coverage,PASS,"{'runs': 20, 'metric_rows': 80, 'threshold_row...","{'runs': 20, 'metric_rows': 80, 'threshold_row...",WARN
7,non-target seeds excluded from metrics,PASS,[],[],ERROR
8,prediction artifacts verified,PASS,20,20,ERROR
9,source model quality clean,WARN,"[{'family': 'lightgcn', 'model_label': 'LightG...",no source WARN runs,WARN


### Notebook 07 multi-seed evaluation: WARN

**Next notebook:** notebook 08 should read this multiseed manifest and build the final tables/figures. If coverage is PARTIAL, rerun notebooks 03-06 with the missing seeds and rerun notebook 07.